# 03 — U-Net (Dice + BCE, augmentation, CUDA)

U-Net (Ronneberger *et al.*, MICCAI 2015) is still the default teaching architecture for biomedical segmentation: a symmetric encoder–decoder with skip connections so fine capillaries can be restored after downsampling.

**Loss.** Dice on its own is unstable when a batch contains almost no vessel. BCE on its own under-emphasises the thin minority class. Adding them (`drive_seg.unet.bce_dice_loss`), restricted to the FOV, is the usual practical recipe.

**Device.** `get_device()` returns CUDA when PyTorch can see an NVIDIA GPU, otherwise CPU. CUDA 12.x is the primary install in `pixi.toml`.


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parents[0] / "src") if Path.cwd().name == "notebooks" else "src")

import pandas as pd
import plotly.express as px
import torch
from torch.utils.data import DataLoader

from drive_seg.data import iter_samples, load_sample, resolve_dataset_root
from drive_seg.device import describe_device, get_device
from drive_seg.metrics import segmentation_metrics
from drive_seg.paths import MODELS_DIR, SYNTHETIC_DIR
from drive_seg.synthetic import generate_synthetic_dataset
from drive_seg.unet import UNet, VesselDataset, bce_dice_loss, predict_mask

if not (SYNTHETIC_DIR / "training" / "images").is_dir():
    generate_synthetic_dataset()

device = get_device()
print(describe_device())
print("cuda built:", torch.backends.cudnn.version() if torch.cuda.is_available() else "n/a")
root = resolve_dataset_root()

## Architecture

Three down / three up blocks, base width 16. That is *deliberately* small so this cell finishes on CPU. For a paper on real DRIVE images, raise `base` to 32–64, train longer, and sample 256² patches from the 584×565 photographs.

In [ ]:
model = UNet(in_ch=1, out_ch=1, base=16)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params/1e6:.2f} M")
x = torch.zeros(1, 1, 256, 256)
print("logits", tuple(model(x).shape))

## Short training loop

Augmentation (`VesselDataset(augment_on=True)`) applies flips, 90° rotations, brightness/contrast jitter and Gaussian noise to the **intensity** only. Masks and the FOV get the same geometric transforms.

In [ ]:
train_ds = VesselDataset(iter_samples(root, split="training"), augment_on=True, seed=0)
loader = DataLoader(train_ds, batch_size=2, shuffle=True)
model = UNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
model.train()
epochs = 8
for epoch in range(1, epochs + 1):
    losses = []
    for images, masks, fovs in loader:
        images, masks, fovs = images.to(device), masks.to(device), fovs.to(device)
        opt.zero_grad(set_to_none=True)
        loss = bce_dice_loss(model(images), masks, fovs)
        loss.backward()
        opt.step()
        losses.append(float(loss.item()))
    history.append({"epoch": epoch, "loss": sum(losses) / len(losses)})
    print(f"epoch {epoch:02d}  loss={history[-1]['loss']:.4f}")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
torch.save({"state_dict": model.state_dict()}, MODELS_DIR / "unet_notebook.pt")
fig = px.line(pd.DataFrame(history), x="epoch", y="loss", markers=True, title="Dice+BCE training loss")
fig.update_layout(template="plotly_white")
fig.show()

## Test Dice

Eight toy images will not match a DRIVE leaderboard. The point is the **recipe**: FOV-aware Dice+BCE, augmentation, CUDA when present. Re-run `pixi run unet -- --epochs 30` on official DRIVE for a serious number.

In [ ]:
model.eval()
rows = []
for s in iter_samples(root, split="test"):
    img, gt, f = load_sample(s)
    pred = predict_mask(model, img, f, device)
    r = segmentation_metrics(pred, gt, f)
    r["sample_id"] = s.sample_id
    rows.append(r)
df = pd.DataFrame(rows)
print(df[["sample_id", "dice", "iou", "sensitivity", "specificity", "precision"]].round(3))
print("mean Dice", round(df.dice.mean(), 3))